# Verify / fix Kilroy protocol consistency

A Kilroy config has three sections: **valve commands** (`<valve_cmd>`), **pump
commands** (`<pump_cmd>`), and **protocols** (`<protocol>`). Each protocol step is a
`<valve>` or `<pump>` element whose *text* names one of those commands. If a step
names a command that isn't defined — because of a typo or an inconsistent name —
Kilroy errors at load.

This notebook:
1. reads a Kilroy config,
2. **verifies** that every protocol step references a defined valve/pump command,
3. proposes fuzzy-matched corrections for any mismatch, and
4. after you confirm, **rewrites** the config in place (backing up to `*.bak`),
   preserving its exact line endings and ISO-8859-1 encoding.

The verify/fix logic lives in `MERci.acquisition.kilroy`
(`check_kilroy_consistency`, `format_consistency_report`, `fix_kilroy_consistency`)
so it can also be called from other code.

In [ ]:
import os
import sys
from pathlib import Path

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.kilroy import (
    load_kilroy_commands,
    iter_protocol_references,
    check_kilroy_consistency,
    format_consistency_report,
    fix_kilroy_consistency,
)

print(f"MERCI_DIR  : {MERCI_DIR}")
print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## Select a Kilroy config

Point `KILROY_CONFIG` at the config to check. By default we look in the
experiment's `settings/` folder (where `prepare_imaging/03` copies the chosen
config) and fall back to the repo's `data/configs/kilroy/` templates. Set the path
explicitly to check any other file.

In [ ]:
# Candidate locations, in priority order.
search_dirs = [SAMPLE_DIR / "settings", MERCI_DIR / "data" / "configs" / "kilroy"]

print("Kilroy configs found:")
found = []
for d in search_dirs:
    for p in sorted(d.glob("kilroy-config-*.xml")):
        found.append(p)
        print(f"  {p}")
if not found:
    print("  (none found - set KILROY_CONFIG manually below)")

# ── Choose the config to verify (edit this line) ───────────────────────────
KILROY_CONFIG = found[0] if found else None
# e.g. KILROY_CONFIG = MERCI_DIR / "data/configs/kilroy/kilroy-config-mf3-hamilton-adaptors-and-direct-260619.xml"

assert KILROY_CONFIG is not None and Path(KILROY_CONFIG).exists(), \
    "Set KILROY_CONFIG to an existing Kilroy config XML."
print(f"\nSelected: {KILROY_CONFIG}")

## Verify consistency

Load the defined commands and check every protocol step against them.

In [ ]:
commands = load_kilroy_commands(KILROY_CONFIG)
references = iter_protocol_references(KILROY_CONFIG)

print(f"Defined valve commands : {len(commands['valve'])}")
print(f"Defined pump commands  : {len(commands['pump'])}")
print(f"Total protocol steps   : {len(references)}")
print()

issues = check_kilroy_consistency(KILROY_CONFIG)
print(format_consistency_report(issues))

## Review and choose fixes

Each mismatch comes with the closest-matching defined command as a *suggestion*:

- **`[normalized]`** — the reference differs from the suggestion only by letter
  case and/or surrounding whitespace. These are almost always safe to apply.
- **others** — the nearest string match by similarity score. Treat these as
  hints, not answers: a low score, or an issue like *`Set Hyb 23` → `Set Hyb 2`*,
  usually means the command is genuinely **missing** from the config (you must add
  the command definition), not that the protocol has a typo. Verify before applying.

The cell below starts `fixes` with **only the safe, normalized matches**. Add
reviewed fuzzy matches by hand, or remove any you don't want.

In [ ]:
# Show every proposal with its confidence.
for i in issues:
    if i.suggestion is None:
        note = "no commands of this kind defined"
    elif i.normalized_match:
        note = "SAFE - case/whitespace only"
    else:
        note = f"REVIEW - nearest match, similarity {i.score:.2f}"
    print(f"[{i.kind}] {i.referenced!r} -> {i.suggestion!r:30}  {note}")

# ── Fixes to apply: (kind, wrong_name) -> correct_name ─────────────────────
# Default: only the safe, normalized matches.
fixes = {
    (i.kind, i.referenced): i.suggestion
    for i in issues
    if i.normalized_match
}

# To also apply a reviewed fuzzy match, add it explicitly, e.g.:
#   fixes[("valve", "SetReadouts")] = "Set Readouts"
# To drop a proposed fix:
#   del fixes[("valve", "Readouts")]

print("\nFixes queued to apply:")
if fixes:
    for (kind, wrong), right in fixes.items():
        print(f"  [{kind}] {wrong!r} -> {right!r}")
else:
    print("  (none)")

## Apply fixes

Rewrites the selected steps in the config file. The original is backed up to
`<config>.bak` before the first change. Any queued fix that matches nothing is
flagged (usually a wrong `kind` or a stale name). A final re-check confirms the
result is clean.

In [ ]:
if fixes:
    total, applied = fix_kilroy_consistency(KILROY_CONFIG, fixes, backup=True)
    print(f"Applied {total} replacement(s).")
    if total:
        print(f"Backup written: {Path(KILROY_CONFIG).name}.bak")
    for kind, wrong, right, n in applied:
        flag = "" if n else "   <-- MATCHED NOTHING (check the name/kind)"
        print(f"  [{kind}] {wrong!r} -> {right!r}: {n} replaced{flag}")

    print("\nRe-check:")
    print(format_consistency_report(check_kilroy_consistency(KILROY_CONFIG)))
else:
    print("No fixes queued - nothing to apply.")